###Setting up to read the data from datalake

In [0]:
scope = 'adventureworkssecret'      

In [0]:
app_id = dbutils.secrets.get(scope=scope, key='app-id')
tenant_id = dbutils.secrets.get(scope=scope, key='tenant-id')
client_secret = dbutils.secrets.get(scope=scope, key='client-secret')
storage_account = dbutils.secrets.get(scope=scope, key='storage-account')

In [0]:
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", app_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

###Data Reading

In [0]:
def read_bronze_table(path):
    return spark.read.format("csv")\
            .option("header", True)\
            .option("inferSchema", True)\
            .load(f"abfss://bronze@adventureworksstorage0.dfs.core.windows.net/{path}")
 

In [0]:
df_products = read_bronze_table('Products')
df_calendars = read_bronze_table('Calendars')
df_customers = read_bronze_table('Customers')
df_product_categories = read_bronze_table('Product Categories')
df_product_subcategories = read_bronze_table('ProductSubCategories')
df_returns = read_bronze_table('Returns')
df_territories = read_bronze_table('Territories')
df_sales_2015 = read_bronze_table('Sales_2015')
df_sales_2016 = read_bronze_table('Sales_2016')
df_sales_2017 = read_bronze_table('Sales_2017')

###Transformations

In [0]:
def write_to_silver_layer(df, path):
    df.write.format("parquet")\
                      .mode("append")\
                      .save(f"abfss://silver@adventureworksstorage0.dfs.core.windows.net/{path}")
    return

####Imports

In [0]:
from pyspark.sql.functions import col, month, year, concat, lit, concat_ws, split, count

###Calendars

In [0]:
df_cal = df_calendars.withColumn('Month', month(col('Date')))\
                     .withColumn('Year', year(col('Date')))
write_to_silver_layer(df_cal, "Calendars")

###Customers

In [0]:
df_customers.display()

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
df_customers.withColumn('FullName', concat(col('Prefix'), lit(' '), col('FirstName'), lit(' '), col('LastName'))).display()

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:440)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:80)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:348)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Attr

In [0]:
 df_cus = df_customers.withColumn('FullName', concat_ws(' ', col('Prefix'), col('FirstName'), col('LastName')))
 write_to_silver_layer(df_cus, 'Customers')

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:440)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:80)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:348)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Attr

###Product subcategories

In [0]:
write_to_silver_layer(df_product_subcategories, 'ProductSubcategories')

###Products

In [0]:
df_products.display()

In [0]:
df_prod = df_products.withColumn("ProductSKU", split(col("ProductSKU"), '-')[0])\
                     .withColumn('ProductName', split(col('ProductName'), ' ')[0])
write_to_silver_layer(df_prod, 'Products')

###Returns

In [0]:
df_returns.display()

In [0]:
write_to_silver_layer(df_returns, "Returns")

###Territories

In [0]:
df_territories.display()

In [0]:
write_to_silver_layer(df_territories, 'Territories')

###Sales

In [0]:
sales_df = df_sales_2015.union(df_sales_2016).union(df_sales_2017)

In [0]:
joined_sales_df = sales_df.join(df_territories, on=sales_df.TerritoryKey == df_territories.SalesTerritoryKey)
write_to_silver_layer(joined_sales_df.select("OrderDate", "StockDate", "OrderNumber", "ProductKey", "CustomerKey", "TerritoryKey", "Region", "Country", "Continent", "OrderLineItem", "OrderQuantity"),"Sales")

In [0]:
# joined_sales_df.groupBy("OrderDate").agg({"OrderNumber": "count"}).show()
joined_sales_df.groupBy("OrderDate").agg(count("OrderNumber").alias("TotalOrders")).display()

In [0]:
df_product_categories.display()

In [0]:
df_territories.display()

###Product Subcategories

In [0]:
write_to_silver_layer(df_product_categories, "ProductCategories")